In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load raw data
matches = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# Basic shape check
print("=== MATCHES ===")
print(f"Rows: {matches.shape[0]}, Columns: {matches.shape[1]}")
print(f"\nColumns: {list(matches.columns)}")

print("\n=== DELIVERIES ===")
print(f"Rows: {deliveries.shape[0]}, Columns: {deliveries.shape[1]}")
print(f"\nColumns: {list(deliveries.columns)}")

# Null check
print("\n=== NULLS IN MATCHES ===")
print(matches.isnull().sum()[matches.isnull().sum() > 0])

print("\n=== NULLS IN DELIVERIES ===")
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])

# Season overview
print("\n=== SEASONS AVAILABLE ===")
print(sorted(matches['season'].unique()))

print("\n=== TOTAL MATCHES PER SEASON ===")
print(matches.groupby('season')['id'].count())

=== MATCHES ===
Rows: 1095, Columns: 20

Columns: ['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

=== DELIVERIES ===
Rows: 260920, Columns: 17

Columns: ['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

=== NULLS IN MATCHES ===
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64

=== NULLS IN DELIVERIES ===
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

=== SEASONS AVAILABLE ===
['2007/08', '2009', '2009/10', '2011', '2012

In [2]:
import os

# ── CLEAN MATCHES ──────────────────────────────────────────────

# 1. Fix inconsistent team names (franchise name changes over the years)
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Pune Warriors': 'Rising Pune Supergiant',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

for col in ['team1', 'team2', 'toss_winner', 'winner']:
    matches[col] = matches[col].replace(team_name_map)

# 2. Remove matches with no result (abandoned/no result)
matches_clean = matches[matches['winner'].notna()].copy()
print(f"Matches after removing no-result: {len(matches_clean)} (removed {len(matches) - len(matches_clean)})")

# 3. Parse date column
matches_clean['date'] = pd.to_datetime(matches_clean['date'])

# 4. Add useful columns
matches_clean['toss_won_match'] = (matches_clean['toss_winner'] == matches_clean['winner']).astype(int)
matches_clean['bat_first_won'] = (
    ((matches_clean['toss_decision'] == 'bat') & (matches_clean['toss_won_match'] == 1)) |
    ((matches_clean['toss_decision'] == 'field') & (matches_clean['toss_won_match'] == 0))
).astype(int)

print("Matches cleaned successfully.")
print(matches_clean[['season','team1','team2','winner','toss_won_match']].head(3))

# ── CLEAN DELIVERIES ──────────────────────────────────────────

# 1. Merge deliver

Matches after removing no-result: 1090 (removed 5)
Matches cleaned successfully.
    season                        team1                  team2  \
0  2007/08  Royal Challengers Bangalore  Kolkata Knight Riders   
1  2007/08                 Punjab Kings    Chennai Super Kings   
2  2007/08               Delhi Capitals       Rajasthan Royals   

                  winner  toss_won_match  
0  Kolkata Knight Riders               0  
1    Chennai Super Kings               1  
2         Delhi Capitals               0  


In [4]:
import os

print("Current directory:", os.getcwd())
print("\n--- Variables in memory ---")
print("matches_clean exists:", 'matches_clean' in globals())
print("deliveries_clean exists:", 'deliveries_clean' in globals())

if 'matches_clean' in globals():
    print("matches_clean rows:", len(matches_clean))
if 'deliveries_clean' in globals():
    print("deliveries_clean rows:", len(deliveries_clean))

print("\n--- Folder check ---")
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/raw', exist_ok=True)
print("processed folder exists:", os.path.exists('../data/processed'))

print("\n--- Files in raw folder ---")
raw_path = '../data/raw'
if os.path.exists(raw_path):
    print(os.listdir(raw_path))
else:
    print("raw folder not found")

Current directory: C:\23261A6720\SPORTS_ANALYSER\notebooks

--- Variables in memory ---
matches_clean exists: True
deliveries_clean exists: False
matches_clean rows: 1090

--- Folder check ---
processed folder exists: True

--- Files in raw folder ---
['deliveries.csv', 'matches.csv']


In [5]:
import pandas as pd
import numpy as np
import sqlite3
import os

print("Step 1: Loading deliveries...")
deliveries = pd.read_csv('../data/raw/deliveries.csv')
print(f"Loaded {len(deliveries)} rows")

print("\nStep 2: Cleaning deliveries...")
team_name_map = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Pune Warriors': 'Rising Pune Supergiant',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

deliveries_clean = deliveries.merge(
    matches_clean[['id','season','date','venue','team1','team2','winner']],
    left_on='match_id', right_on='id', how='inner'
)
for col in ['batting_team', 'bowling_team']:
    deliveries_clean[col] = deliveries_clean[col].replace(team_name_map)

def get_phase(over):
    if over <= 6:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'

deliveries_clean['phase'] = deliveries_clean['over'].apply(get_phase)
deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('not_out')
deliveries_clean['dismissal_kind']   = deliveries_clean['dismissal_kind'].fillna('not_out')
deliveries_clean['extras_type']      = deliveries_clean['extras_type'].fillna('none')
print(f"Deliveries clean: {len(deliveries_clean)} rows")

print("\nStep 3: Saving cleaned CSVs...")
matches_clean.to_csv('../data/processed/matches_clean.csv', index=False)
deliveries_clean.to_csv('../data/processed/deliveries_clean.csv', index=False)
print("✓ matches_clean.csv saved")
print("✓ deliveries_clean.csv saved")

print("\nStep 4: Building batter stats...")
batter_match = deliveries_clean.groupby(
    ['match_id','season','date','venue','batting_team','batter']
).agg(
    runs        =('batsman_runs','sum'),
    balls_faced =('batsman_runs','count'),
    fours       =('batsman_runs', lambda x: (x==4).sum()),
    sixes       =('batsman_runs', lambda x: (x==6).sum()),
).reset_index()

batter_match['strike_rate'] = (batter_match['runs'] / batter_match['balls_faced'] * 100).round(1)
batter_match['fantasy_pts'] = (
    batter_match['runs'] * 1 +
    batter_match['fours'] * 1 +
    batter_match['sixes'] * 2 +
    (batter_match['runs'] >= 30).astype(int) * 4 +
    (batter_match['runs'] >= 50).astype(int) * 8 +
    (batter_match['runs'] >= 100).astype(int) * 16 +
    (batter_match['runs'] == 0).astype(int) * -2
)
batter_match = batter_match.sort_values('date')
batter_match['form_score'] = (
    batter_match.groupby('batter')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"✓ Batter records: {len(batter_match)}")

print("\nStep 5: Building bowler stats...")
legal = deliveries_clean[deliveries_clean['extras_type'] != 'wides']
bowler_match = legal.groupby(
    ['match_id','season','date','venue','bowling_team','bowler']
).agg(
    balls_bowled =('ball','count'),
    runs_given   =('total_runs','sum'),
    wickets      =('is_wicket','sum'),
).reset_index()

bowler_match['overs']   = (bowler_match['balls_bowled'] / 6).round(1)
bowler_match['economy'] = (bowler_match['runs_given'] / bowler_match['overs']).round(2)
bowler_match['economy'] = bowler_match['economy'].replace([float('inf'), float('nan')], 0)
bowler_match['fantasy_pts'] = (
    bowler_match['wickets'] * 25 +
    (bowler_match['wickets'] >= 3).astype(int) * 4 +
    (bowler_match['wickets'] >= 5).astype(int) * 8 +
    (bowler_match['economy'] < 5).astype(int) * 6 +
    (bowler_match['economy'] > 9).astype(int) * -2
)
bowler_match['form_score'] = (
    bowler_match.sort_values('date')
    .groupby('bowler')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"✓ Bowler records: {len(bowler_match)}")

print("\nStep 6: Building team phase stats...")
team_match = deliveries_clean.groupby(
    ['match_id','season','batting_team','phase']
).agg(
    runs    =('batsman_runs','sum'),
    wickets =('is_wicket','sum'),
    balls   =('ball','count'),
).reset_index()
team_match['run_rate'] = (team_match['runs'] / (team_match['balls'] / 6)).round(2)
print(f"✓ Team phase records: {len(team_match)}")

print("\nStep 7: Building venue stats...")
venue_stats = deliveries_clean.groupby(['venue','inning']).agg(
    avg_runs    =('batsman_runs','sum'),
    avg_wickets =('is_wicket','sum'),
    matches     =('match_id','nunique'),
).reset_index()
venue_stats['avg_runs_per_match']    = (venue_stats['avg_runs'] / venue_stats['matches']).round(1)
venue_stats['avg_wickets_per_match'] = (venue_stats['avg_wickets'] / venue_stats['matches']).round(1)
print(f"✓ Venue records: {len(venue_stats)}")

print("\nStep 8: Saving all feature CSVs...")
batter_match.to_csv('../data/processed/batter_stats.csv', index=False)
bowler_match.to_csv('../data/processed/bowler_stats.csv', index=False)
team_match.to_csv('../data/processed/team_phase_stats.csv', index=False)
venue_stats.to_csv('../data/processed/venue_stats.csv', index=False)
print("✓ All 4 feature CSVs saved")

print("\nStep 9: Loading into SQLite...")
conn = sqlite3.connect('../data/processed/ipl.db')
batter_match.to_sql('batter_stats',     conn, if_exists='replace', index=False)
bowler_match.to_sql('bowler_stats',     conn, if_exists='replace', index=False)
team_match.to_sql('team_phase_stats',   conn, if_exists='replace', index=False)
venue_stats.to_sql('venue_stats',       conn, if_exists='replace', index=False)
matches_clean.to_sql('matches',         conn, if_exists='replace', index=False)

print("\n=== FINAL VALIDATION ===")
checks = {
    "Batter records" : "SELECT COUNT(*) FROM batter_stats",
    "Bowler records" : "SELECT COUNT(*) FROM bowler_stats",
    "Seasons"        : "SELECT COUNT(DISTINCT season) FROM matches",
    "Top scorer"     : "SELECT batter, SUM(runs) as r FROM batter_stats GROUP BY batter ORDER BY r DESC LIMIT 1",
    "Top wicket"     : "SELECT bowler, SUM(wickets) as w FROM bowler_stats GROUP BY bowler ORDER BY w DESC LIMIT 1",
}
for label, query in checks.items():
    result = pd.read_sql(query, conn)
    vals = " | ".join(str(v) for v in result.iloc[0].values)
    print(f"{label}: {vals}")

conn.close()
print("\n✓ ipl.db saved")
print("\n🎉 PHASE 1 COMPLETE — All data ready for Power BI and Streamlit")

Step 1: Loading deliveries...
Loaded 260920 rows

Step 2: Cleaning deliveries...
Deliveries clean: 260430 rows

Step 3: Saving cleaned CSVs...
✓ matches_clean.csv saved
✓ deliveries_clean.csv saved

Step 4: Building batter stats...
✓ Batter records: 16475

Step 5: Building bowler stats...
✓ Bowler records: 12942

Step 6: Building team phase stats...
✓ Team phase records: 6389

Step 7: Building venue stats...
✓ Venue records: 140

Step 8: Saving all feature CSVs...
✓ All 4 feature CSVs saved

Step 9: Loading into SQLite...

=== FINAL VALIDATION ===
Batter records: 16475
Bowler records: 12942
Seasons: 17
Top scorer: V Kohli | 7987
Top wicket: DJ Bravo | 206

✓ ipl.db saved

🎉 PHASE 1 COMPLETE — All data ready for Power BI and Streamlit


In [6]:
import pandas as pd

# Reload and fix all CSVs
files = ['batter_stats', 'bowler_stats', 'team_phase_stats']

for name in files:
    df = pd.read_csv(f'../data/processed/{name}.csv')
    if 'season' in df.columns:
        df['season'] = df['season'].astype(str).str.replace('/', '-', regex=False)
    df.to_csv(f'../data/processed/{name}.csv', index=False)
    print(f"✓ {name}.csv fixed — season format updated")

# Verify
test = pd.read_csv('../data/processed/batter_stats.csv')
print("\nSeason values now:", test['season'].unique()[:5])
print("Total rows:", len(test))

✓ batter_stats.csv fixed — season format updated
✓ bowler_stats.csv fixed — season format updated
✓ team_phase_stats.csv fixed — season format updated

Season values now: <ArrowStringArray>
['2007-08', '2009', '2009-10', '2011', '2012']
Length: 5, dtype: str
Total rows: 16475


In [7]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print("=" * 55)
print("MASTER DATA PREPARATION — POWER BI READY")
print("=" * 55)

# ── LOAD RAW FILES ─────────────────────────────────────────
print("\n[1/9] Loading raw files...")
matches   = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')
print(f"      matches: {len(matches)} rows | deliveries: {len(deliveries)} rows")

# ── CLEAN MATCHES ──────────────────────────────────────────
print("\n[2/9] Cleaning matches...")
team_map = {
    'Delhi Daredevils'      : 'Delhi Capitals',
    'Deccan Chargers'       : 'Sunrisers Hyderabad',
    'Pune Warriors'         : 'Rising Pune Supergiant',
    'Kings XI Punjab'       : 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

matches_clean = matches[matches['winner'].notna()].copy()
matches_clean['date'] = pd.to_datetime(matches_clean['date']).dt.strftime('%Y-%m-%d')

# FIX: season — replace / with - so Power BI reads as Text not Date
matches_clean['season'] = matches_clean['season'].astype(str).str.replace('/', '-', regex=False)

matches_clean['toss_won_match'] = (matches_clean['toss_winner'] == matches_clean['winner']).astype(int)
print(f"      Clean matches: {len(matches_clean)}")

# ── CLEAN DELIVERIES ───────────────────────────────────────
print("\n[3/9] Cleaning deliveries...")
deliveries_clean = deliveries.merge(
    matches_clean[['id','season','date','venue','team1','team2','winner']],
    left_on='match_id', right_on='id', how='inner'
)
for col in ['batting_team','bowling_team']:
    deliveries_clean[col] = deliveries_clean[col].replace(team_map)

deliveries_clean['phase'] = deliveries_clean['over'].apply(
    lambda o: 'Powerplay' if o <= 6 else ('Middle' if o <= 15 else 'Death')
)
deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('not_out')
deliveries_clean['dismissal_kind']   = deliveries_clean['dismissal_kind'].fillna('not_out')
deliveries_clean['extras_type']      = deliveries_clean['extras_type'].fillna('none')
print(f"      Clean deliveries: {len(deliveries_clean)}")

# ── BATTER STATS ───────────────────────────────────────────
print("\n[4/9] Building batter stats...")
batter = deliveries_clean.groupby(
    ['match_id','season','date','venue','batting_team','batter']
).agg(
    runs        =('batsman_runs','sum'),
    balls_faced =('batsman_runs','count'),
    fours       =('batsman_runs', lambda x: (x==4).sum()),
    sixes       =('batsman_runs', lambda x: (x==6).sum()),
).reset_index()

batter['strike_rate'] = np.where(
    batter['balls_faced'] > 0,
    (batter['runs'] / batter['balls_faced'] * 100).round(1), 0
)
batter['fantasy_pts'] = (
    batter['runs'] * 1 +
    batter['fours'] * 1 +
    batter['sixes'] * 2 +
    (batter['runs'] >= 30).astype(int) * 4 +
    (batter['runs'] >= 50).astype(int) * 8 +
    (batter['runs'] >= 100).astype(int) * 16 +
    (batter['runs'] == 0).astype(int) * -2
)
batter = batter.sort_values(['batter','date']).reset_index(drop=True)
batter['form_score'] = (
    batter.groupby('batter')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"      Batter records: {len(batter)}")

# ── BOWLER STATS ───────────────────────────────────────────
print("\n[5/9] Building bowler stats...")
legal = deliveries_clean[~deliveries_clean['extras_type'].isin(['wides','noballs'])]
bowler = legal.groupby(
    ['match_id','season','date','venue','bowling_team','bowler']
).agg(
    balls_bowled =('ball','count'),
    runs_given   =('total_runs','sum'),
    wickets      =('is_wicket','sum'),
).reset_index()

bowler['overs']   = (bowler['balls_bowled'] / 6).round(1)
bowler['economy'] = np.where(
    bowler['overs'] > 0,
    (bowler['runs_given'] / bowler['overs']).round(2), 0
)
bowler['fantasy_pts'] = (
    bowler['wickets'] * 25 +
    (bowler['wickets'] >= 3).astype(int) * 4 +
    (bowler['wickets'] >= 5).astype(int) * 8 +
    (bowler['economy'] < 5).astype(int) * 6 +
    (bowler['economy'] > 9).astype(int) * -2
)
bowler = bowler.sort_values(['bowler','date']).reset_index(drop=True)
bowler['form_score'] = (
    bowler.groupby('bowler')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"      Bowler records: {len(bowler)}")

# ── TEAM PHASE STATS ───────────────────────────────────────
print("\n[6/9] Building team phase stats...")
team_phase = deliveries_clean.groupby(
    ['match_id','season','batting_team','phase']
).agg(
    runs    =('batsman_runs','sum'),
    wickets =('is_wicket','sum'),
    balls   =('ball','count'),
).reset_index()
team_phase['run_rate'] = np.where(
    team_phase['balls'] > 0,
    (team_phase['runs'] / (team_phase['balls'] / 6)).round(2), 0
)
print(f"      Team phase records: {len(team_phase)}")

# ── VENUE STATS ────────────────────────────────────────────
print("\n[7/9] Building venue stats...")
venue_stats = deliveries_clean.groupby(['venue','inning']).agg(
    total_runs    =('batsman_runs','sum'),
    total_wickets =('is_wicket','sum'),
    matches       =('match_id','nunique'),
).reset_index()
venue_stats['avg_runs_per_match']    = (venue_stats['total_runs'] / venue_stats['matches']).round(1)
venue_stats['avg_wickets_per_match'] = (venue_stats['total_wickets'] / venue_stats['matches']).round(1)
print(f"      Venue records: {len(venue_stats)}")

# ── DIMENSION TABLES (fixes many-to-many) ──────────────────
print("\n[8/9] Building dimension tables...")

# DIM_MATCH — one row per match (the "one" side)
dim_match = matches_clean[['id','season','date','venue','team1','team2','winner']].copy()
dim_match = dim_match.rename(columns={'id':'match_id'})
dim_match = dim_match.drop_duplicates(subset='match_id').reset_index(drop=True)

# DIM_TEAM — one row per team
all_teams = pd.concat([
    batter[['batting_team']].rename(columns={'batting_team':'team_name'}),
    bowler[['bowling_team']].rename(columns={'bowling_team':'team_name'})
]).drop_duplicates().dropna().sort_values('team_name').reset_index(drop=True)
all_teams['team_id'] = range(1, len(all_teams)+1)

# DIM_VENUE — one row per venue
dim_venue = pd.DataFrame({
    'venue'    : deliveries_clean['venue'].dropna().unique()
}).sort_values('venue').reset_index(drop=True)
dim_venue['venue_id'] = range(1, len(dim_venue)+1)

# DIM_SEASON — one row per season
dim_season = pd.DataFrame({
    'season': sorted(batter['season'].dropna().unique())
}).reset_index(drop=True)
dim_season['season_id'] = range(1, len(dim_season)+1)

# DIM_PLAYER — one row per player
all_players = pd.concat([
    batter[['batter']].rename(columns={'batter':'player_name'}),
    bowler[['bowler']].rename(columns={'bowler':'player_name'})
]).drop_duplicates().sort_values('player_name').reset_index(drop=True)
all_players['player_id'] = range(1, len(all_players)+1)

print(f"      dim_match:  {len(dim_match)} | dim_team: {len(all_teams)}")
print(f"      dim_venue:  {len(dim_venue)} | dim_season: {len(dim_season)}")
print(f"      dim_player: {len(all_players)}")

# ── SAVE EVERYTHING ────────────────────────────────────────
print("\n[9/9] Saving all files...")
os.makedirs('../data/processed', exist_ok=True)

# Fact tables
batter.to_csv('../data/processed/batter_stats.csv',         index=False)
bowler.to_csv('../data/processed/bowler_stats.csv',         index=False)
team_phase.to_csv('../data/processed/team_phase_stats.csv', index=False)
venue_stats.to_csv('../data/processed/venue_stats.csv',     index=False)
matches_clean.to_csv('../data/processed/matches_clean.csv', index=False)

# Dimension tables
dim_match.to_csv('../data/processed/dim_match.csv',     index=False)
all_teams.to_csv('../data/processed/dim_team.csv',      index=False)
dim_venue.to_csv('../data/processed/dim_venue.csv',     index=False)
dim_season.to_csv('../data/processed/dim_season.csv',   index=False)
all_players.to_csv('../data/processed/dim_player.csv',  index=False)

# SQLite
conn = sqlite3.connect('../data/processed/ipl.db')
for df, name in [
    (batter,       'batter_stats'),
    (bowler,       'bowler_stats'),
    (team_phase,   'team_phase_stats'),
    (venue_stats,  'venue_stats'),
    (matches_clean,'matches'),
    (dim_match,    'dim_match'),
    (all_teams,    'dim_team'),
    (dim_venue,    'dim_venue'),
    (dim_season,   'dim_season'),
    (all_players,  'dim_player'),
]:
    df.to_sql(name, conn, if_exists='replace', index=False)
conn.close()

# ── FINAL CHECKS ───────────────────────────────────────────
print("\n" + "=" * 55)
print("VALIDATION")
print("=" * 55)
print(f"batter_stats    : {len(batter):>6} rows | columns: {list(batter.columns)}")
print(f"bowler_stats    : {len(bowler):>6} rows | columns: {list(bowler.columns)}")
print(f"team_phase_stats: {len(team_phase):>6} rows")
print(f"venue_stats     : {len(venue_stats):>6} rows")
print(f"dim_match       : {len(dim_match):>6} rows (unique match_ids)")
print(f"dim_team        : {len(all_teams):>6} rows (unique teams)")
print(f"dim_season      : {len(dim_season):>6} rows | values: {list(dim_season['season'])}")

# Confirm no slash in season
assert '/' not in str(batter['season'].iloc[0]), "Season still has slash!"
print("\n✓ Season format clean — no slashes")
print("✓ SQLite DB updated")
print("✓ All CSVs saved")

print("\n" + "=" * 55)
print("ALL DONE — NOW OPEN POWER BI FRESH")
print("=" * 55)
print("""
POWER BI LOAD ORDER:
1. Close Power BI completely
2. Open fresh → New report
3. Get Data → Text/CSV → load in this order:

   DIMENSION TABLES FIRST:
   □ dim_match.csv
   □ dim_team.csv
   □ dim_venue.csv
   □ dim_season.csv
   □ dim_player.csv

   FACT TABLES SECOND:
   □ batter_stats.csv
   □ bowler_stats.csv
   □ team_phase_stats.csv
   □ venue_stats.csv

4. In Power Query — for EVERY table:
   • Select 'season' column → Data Type → TEXT
   • Select 'date' column → Data Type → DATE
   • Click Close & Apply

5. RELATIONSHIPS (Model view):
   dim_match[match_id]   → batter_stats[match_id]    ONE-to-MANY
   dim_match[match_id]   → bowler_stats[match_id]    ONE-to-MANY
   dim_match[venue]      → venue_stats[venue]         ONE-to-MANY
   dim_season[season]    → batter_stats[season]       ONE-to-MANY
   dim_season[season]    → bowler_stats[season]       ONE-to-MANY
   dim_team[team_name]   → batter_stats[batting_team] ONE-to-MANY
   dim_team[team_name]   → bowler_stats[bowling_team] ONE-to-MANY
""")

MASTER DATA PREPARATION — POWER BI READY

[1/9] Loading raw files...
      matches: 1095 rows | deliveries: 260920 rows

[2/9] Cleaning matches...
      Clean matches: 1090

[3/9] Cleaning deliveries...
      Clean deliveries: 260430

[4/9] Building batter stats...
      Batter records: 16475

[5/9] Building bowler stats...
      Bowler records: 12942

[6/9] Building team phase stats...
      Team phase records: 6389

[7/9] Building venue stats...
      Venue records: 140

[8/9] Building dimension tables...
      dim_match:  1090 | dim_team: 14
      dim_venue:  58 | dim_season: 17
      dim_player: 731

[9/9] Saving all files...

VALIDATION
batter_stats    :  16475 rows | columns: ['match_id', 'season', 'date', 'venue', 'batting_team', 'batter', 'runs', 'balls_faced', 'fours', 'sixes', 'strike_rate', 'fantasy_pts', 'form_score']
bowler_stats    :  12942 rows | columns: ['match_id', 'season', 'date', 'venue', 'bowling_team', 'bowler', 'balls_bowled', 'runs_given', 'wickets', 'overs', '

In [8]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print("="*55)
print("FINAL MASTER SCRIPT — CLEAN POWER BI STAR SCHEMA")
print("="*55)

# ── LOAD RAW ──────────────────────────────────────────────
matches    = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')

# ── CLEAN ─────────────────────────────────────────────────
team_map = {
    'Delhi Daredevils'       : 'Delhi Capitals',
    'Deccan Chargers'        : 'Sunrisers Hyderabad',
    'Pune Warriors'          : 'Rising Pune Supergiant',
    'Kings XI Punjab'        : 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

matches_clean = matches[matches['winner'].notna()].copy()
matches_clean['date']   = pd.to_datetime(matches_clean['date']).dt.strftime('%Y-%m-%d')
matches_clean['season'] = matches_clean['season'].astype(str).str.replace('/','-',regex=False)

deliveries_clean = deliveries.merge(
    matches_clean[['id','season','date','venue','team1','team2','winner']],
    left_on='match_id', right_on='id', how='inner'
)
for col in ['batting_team','bowling_team']:
    deliveries_clean[col] = deliveries_clean[col].replace(team_map)

deliveries_clean['phase'] = deliveries_clean['over'].apply(
    lambda o: 'Powerplay' if o<=6 else ('Middle' if o<=15 else 'Death')
)
deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('not_out')
deliveries_clean['dismissal_kind']   = deliveries_clean['dismissal_kind'].fillna('not_out')
deliveries_clean['extras_type']      = deliveries_clean['extras_type'].fillna('none')

# ══════════════════════════════════════════════════════════
# BUILD EXACTLY 4 FLAT TABLES — NO RELATIONSHIPS NEEDED
# Power BI will use slicers directly on each table
# This completely eliminates all relationship errors
# ══════════════════════════════════════════════════════════

# ── TABLE 1: BATTER SUMMARY ───────────────────────────────
# One row per player per match — fully denormalized
print("\n[1/4] Building batter_summary...")
b = deliveries_clean.groupby(
    ['match_id','season','date','venue','batting_team','batter']
).agg(
    runs        =('batsman_runs','sum'),
    balls_faced =('batsman_runs','count'),
    fours       =('batsman_runs', lambda x:(x==4).sum()),
    sixes       =('batsman_runs', lambda x:(x==6).sum()),
).reset_index()

b['strike_rate']  = np.where(b['balls_faced']>0, (b['runs']/b['balls_faced']*100).round(1), 0)
b['is_fifty']     = (b['runs']>=50).astype(int)
b['is_hundred']   = (b['runs']>=100).astype(int)
b['is_duck']      = (b['runs']==0).astype(int)
b['fantasy_pts']  = (
    b['runs']*1 + b['fours']*1 + b['sixes']*2 +
    (b['runs']>=30).astype(int)*4 +
    b['is_fifty']*8 + b['is_hundred']*16 +
    b['is_duck']*-2
)
b = b.sort_values(['batter','date']).reset_index(drop=True)
b['form_score'] = (
    b.groupby('batter')['fantasy_pts']
    .transform(lambda x: x.rolling(5,min_periods=1).mean())
    .round(1)
)
# Add career aggregates per player as extra columns
career = b.groupby('batter').agg(
    career_runs    =('runs','sum'),
    career_matches =('match_id','nunique'),
    career_avg     =('runs','mean'),
    career_sr      =('strike_rate','mean'),
    career_fifties =('is_fifty','sum'),
    career_hundreds=('is_hundred','sum'),
).round(1).reset_index()
b = b.merge(career, on='batter', how='left')
print(f"    batter_summary: {len(b)} rows, {len(b.columns)} columns")

# ── TABLE 2: BOWLER SUMMARY ───────────────────────────────
print("[2/4] Building bowler_summary...")
legal = deliveries_clean[~deliveries_clean['extras_type'].isin(['wides','noballs'])]
w = legal.groupby(
    ['match_id','season','date','venue','bowling_team','bowler']
).agg(
    balls_bowled =('ball','count'),
    runs_given   =('total_runs','sum'),
    wickets      =('is_wicket','sum'),
).reset_index()

w['overs']        = (w['balls_bowled']/6).round(1)
w['economy']      = np.where(w['overs']>0, (w['runs_given']/w['overs']).round(2), 0)
w['is_3fer']      = (w['wickets']>=3).astype(int)
w['is_5fer']      = (w['wickets']>=5).astype(int)
w['fantasy_pts']  = (
    w['wickets']*25 + w['is_3fer']*4 + w['is_5fer']*8 +
    (w['economy']<5).astype(int)*6 +
    (w['economy']>9).astype(int)*-2
)
w = w.sort_values(['bowler','date']).reset_index(drop=True)
w['form_score'] = (
    w.groupby('bowler')['fantasy_pts']
    .transform(lambda x: x.rolling(5,min_periods=1).mean())
    .round(1)
)
career_b = w.groupby('bowler').agg(
    career_wickets  =('wickets','sum'),
    career_matches  =('match_id','nunique'),
    career_economy  =('economy','mean'),
    career_3fers    =('is_3fer','sum'),
    career_5fers    =('is_5fer','sum'),
).round(2).reset_index()
w = w.merge(career_b, on='bowler', how='left')
print(f"    bowler_summary: {len(w)} rows, {len(w.columns)} columns")

# ── TABLE 3: TEAM PHASE SUMMARY ───────────────────────────
print("[3/4] Building team_phase_summary...")
tp = deliveries_clean.groupby(
    ['match_id','season','batting_team','phase']
).agg(
    runs    =('batsman_runs','sum'),
    wickets =('is_wicket','sum'),
    balls   =('ball','count'),
).reset_index()
tp['run_rate'] = np.where(tp['balls']>0, (tp['runs']/(tp['balls']/6)).round(2), 0)

# Add team career phase averages
team_avg = tp.groupby(['batting_team','phase']).agg(
    avg_runs_in_phase =('runs','mean'),
    avg_rr_in_phase   =('run_rate','mean'),
).round(2).reset_index()
tp = tp.merge(team_avg, on=['batting_team','phase'], how='left')
print(f"    team_phase_summary: {len(tp)} rows, {len(tp.columns)} columns")

# ── TABLE 4: VENUE SUMMARY ────────────────────────────────
print("[4/4] Building venue_summary...")
vs = deliveries_clean.groupby(['venue','inning']).agg(
    total_runs    =('batsman_runs','sum'),
    total_wickets =('is_wicket','sum'),
    matches       =('match_id','nunique'),
).reset_index()
vs['avg_runs_per_match']    = (vs['total_runs']/vs['matches']).round(1)
vs['avg_wickets_per_match'] = (vs['total_wickets']/vs['matches']).round(1)
print(f"    venue_summary: {len(vs)} rows, {len(vs.columns)} columns")

# ── SAVE ──────────────────────────────────────────────────
print("\nSaving files...")
os.makedirs('../data/processed', exist_ok=True)

b.to_csv('../data/processed/batter_summary.csv',     index=False)
w.to_csv('../data/processed/bowler_summary.csv',     index=False)
tp.to_csv('../data/processed/team_phase_summary.csv',index=False)
vs.to_csv('../data/processed/venue_summary.csv',     index=False)

# Also save SQLite
conn = sqlite3.connect('../data/processed/ipl.db')
b.to_sql('batter_summary',      conn, if_exists='replace', index=False)
w.to_sql('bowler_summary',      conn, if_exists='replace', index=False)
tp.to_sql('team_phase_summary', conn, if_exists='replace', index=False)
vs.to_sql('venue_summary',      conn, if_exists='replace', index=False)
conn.close()

# ── VALIDATE ──────────────────────────────────────────────
print("\n" + "="*55)
print("VALIDATION")
print("="*55)
print(f"batter_summary      : {len(b):>6} rows")
print(f"bowler_summary      : {len(w):>6} rows")
print(f"team_phase_summary  : {len(tp):>6} rows")
print(f"venue_summary       : {len(vs):>6} rows")
print(f"\nTop career scorer   : {career.nlargest(1,'career_runs')['batter'].values[0]} — {career.nlargest(1,'career_runs')['career_runs'].values[0]} runs")
print(f"Top career wickets  : {career_b.nlargest(1,'career_wickets')['bowler'].values[0]} — {career_b.nlargest(1,'career_wickets')['career_wickets'].values[0]} wickets")
print(f"Seasons covered     : {sorted(b['season'].unique())}")
print(f"Teams covered       : {sorted(b['batting_team'].unique())}")

print("\n" + "="*55)
print("FILES READY — OPEN POWER BI NOW")
print("="*55)
print("""
EXACT STEPS IN POWER BI — NO ERRORS:

1. Open Power BI Desktop → New file

2. Get Data → Text/CSV → load ONLY these 4 files:
   • batter_summary.csv
   • bowler_summary.csv
   • team_phase_summary.csv
   • venue_summary.csv

3. In Power Query for EACH file:
   • Click 'season' column → Transform →
     Data Type → Text → Replace Current
   • Click 'date' column → Data Type →
     Date → Replace Current
   • Click Close & Apply

4. DO NOT CREATE ANY RELATIONSHIPS
   These are flat tables — no joins needed
   Each table works independently with slicers

5. Build visuals using slicers to filter:
   • Season slicer filters batter_summary directly
   • Team slicer filters batter_summary directly
   • No cross-table filtering needed

ZERO relationship errors guaranteed.
""")

FINAL MASTER SCRIPT — CLEAN POWER BI STAR SCHEMA

[1/4] Building batter_summary...
    batter_summary: 16475 rows, 22 columns
[2/4] Building bowler_summary...
    bowler_summary: 12942 rows, 20 columns
[3/4] Building team_phase_summary...
    team_phase_summary: 6389 rows, 10 columns
[4/4] Building venue_summary...
    venue_summary: 140 rows, 7 columns

Saving files...

VALIDATION
batter_summary      :  16475 rows
bowler_summary      :  12942 rows
team_phase_summary  :   6389 rows
venue_summary       :    140 rows

Top career scorer   : V Kohli — 7987 runs
Top career wickets  : DJ Bravo — 206 wickets
Seasons covered     : ['2007-08', '2009', '2009-10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020-21', '2021', '2022', '2023', '2024']
Teams covered       : ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("IPL INTELLIGENCE PLATFORM — MASTER BUILD SCRIPT")
print("="*60)

# ═══════════════════════════════════════════════════════════════
# SECTION 1 — LOAD & CLEAN
# ═══════════════════════════════════════════════════════════════
print("\n── SECTION 1: LOADING RAW DATA ──")
matches    = pd.read_csv('../data/raw/matches.csv')
deliveries = pd.read_csv('../data/raw/deliveries.csv')
print(f"matches: {len(matches)} rows | deliveries: {len(deliveries)} rows")

# Fix team names that changed over the years
team_map = {
    'Delhi Daredevils'       : 'Delhi Capitals',
    'Deccan Chargers'        : 'Sunrisers Hyderabad',
    'Pune Warriors'          : 'Rising Pune Supergiant',
    'Kings XI Punjab'        : 'Punjab Kings',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}

for col in ['team1','team2','toss_winner','winner']:
    matches[col] = matches[col].replace(team_map)

# Remove abandoned matches (no winner)
matches_clean = matches[matches['winner'].notna()].copy()

# KEY FIX: season uses dash not slash — Power BI reads 2007/08 as a date
# 2007-08 stays as text — no more errors
matches_clean['season'] = (matches_clean['season']
                           .astype(str)
                           .str.replace('/', '-', regex=False))

# Date as proper string YYYY-MM-DD — Power BI understands this
matches_clean['date'] = (pd.to_datetime(matches_clean['date'])
                         .dt.strftime('%Y-%m-%d'))

# Useful match-level columns
matches_clean['toss_won_match'] = (
    matches_clean['toss_winner'] == matches_clean['winner']
).astype(int)

print(f"matches_clean: {len(matches_clean)} rows (removed {len(matches)-len(matches_clean)} abandoned)")

# Merge deliveries with match metadata — this gives every ball its
# season, date, venue, teams — everything in one place
deliveries_clean = deliveries.merge(
    matches_clean[['id','season','date','venue','team1','team2','winner']],
    left_on='match_id', right_on='id', how='inner'
)

for col in ['batting_team','bowling_team']:
    deliveries_clean[col] = deliveries_clean[col].replace(team_map)

# Phase — every ball now knows which phase of the match it belongs to
deliveries_clean['phase'] = deliveries_clean['over'].apply(
    lambda o: 'Powerplay' if o <= 6 else ('Middle' if o <= 15 else 'Death')
)

# Fill nulls — these are NOT errors, just balls where nobody got out
deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('not_out')
deliveries_clean['dismissal_kind']   = deliveries_clean['dismissal_kind'].fillna('not_out')
deliveries_clean['extras_type']      = deliveries_clean['extras_type'].fillna('none')

print(f"deliveries_clean: {len(deliveries_clean)} rows")

# ═══════════════════════════════════════════════════════════════
# SECTION 2 — FACT TABLES
# (These go in the CENTRE of the star schema)
# ═══════════════════════════════════════════════════════════════
print("\n── SECTION 2: BUILDING FACT TABLES ──")

# ── FACT 1: BATTER STATS ──────────────────────────────────────
# One row per batter per match
batter_stats = deliveries_clean.groupby(
    ['match_id','season','date','venue','batting_team','batter']
).agg(
    runs        = ('batsman_runs', 'sum'),
    balls_faced = ('batsman_runs', 'count'),
    fours       = ('batsman_runs', lambda x: (x==4).sum()),
    sixes       = ('batsman_runs', lambda x: (x==6).sum()),
).reset_index()

# Derived columns — these are what Power BI will visualize
batter_stats['strike_rate'] = np.where(
    batter_stats['balls_faced'] > 0,
    (batter_stats['runs'] / batter_stats['balls_faced'] * 100).round(1), 0
)
batter_stats['is_fifty']   = (batter_stats['runs'] >= 50).astype(int)
batter_stats['is_hundred'] = (batter_stats['runs'] >= 100).astype(int)
batter_stats['is_duck']    = (batter_stats['runs'] == 0).astype(int)

# Fantasy points — Dream11 standard scoring system
# This is the core metric of the entire project
batter_stats['fantasy_pts'] = (
    batter_stats['runs'] * 1          +  # 1 pt per run
    batter_stats['fours'] * 1         +  # 1 pt per four
    batter_stats['sixes'] * 2         +  # 2 pts per six
    (batter_stats['runs']>=30).astype(int) * 4  +  # 30-run bonus
    batter_stats['is_fifty'] * 8      +  # 50-run bonus
    batter_stats['is_hundred'] * 16   +  # century bonus
    batter_stats['is_duck'] * -2         # duck penalty
)

# Form score — rolling average of last 5 matches
# This is what makes our tool better than looking at season averages
batter_stats = batter_stats.sort_values(['batter','date']).reset_index(drop=True)
batter_stats['form_score'] = (
    batter_stats.groupby('batter')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"batter_stats: {len(batter_stats)} rows")

# ── FACT 2: BOWLER STATS ──────────────────────────────────────
# Exclude wides and no balls for accurate over calculation
legal = deliveries_clean[
    ~deliveries_clean['extras_type'].isin(['wides','noballs'])
]

bowler_stats = legal.groupby(
    ['match_id','season','date','venue','bowling_team','bowler']
).agg(
    balls_bowled = ('ball', 'count'),
    runs_given   = ('total_runs', 'sum'),
    wickets      = ('is_wicket', 'sum'),
).reset_index()

bowler_stats['overs']   = (bowler_stats['balls_bowled'] / 6).round(1)
bowler_stats['economy'] = np.where(
    bowler_stats['overs'] > 0,
    (bowler_stats['runs_given'] / bowler_stats['overs']).round(2), 0
)
bowler_stats['is_3fer'] = (bowler_stats['wickets'] >= 3).astype(int)
bowler_stats['is_5fer'] = (bowler_stats['wickets'] >= 5).astype(int)

bowler_stats['fantasy_pts'] = (
    bowler_stats['wickets'] * 25      +  # 25 pts per wicket
    bowler_stats['is_3fer'] * 4       +  # 3-wicket bonus
    bowler_stats['is_5fer'] * 8       +  # 5-wicket bonus
    (bowler_stats['economy']<5).astype(int) * 6   +  # economy bonus
    (bowler_stats['economy']>9).astype(int) * -2     # economy penalty
)

bowler_stats = bowler_stats.sort_values(['bowler','date']).reset_index(drop=True)
bowler_stats['form_score'] = (
    bowler_stats.groupby('bowler')['fantasy_pts']
    .transform(lambda x: x.rolling(5, min_periods=1).mean())
    .round(1)
)
print(f"bowler_stats: {len(bowler_stats)} rows")

# ── FACT 3: TEAM PHASE STATS ─────────────────────────────────
# How each team performs in each phase — powerplay / middle / death
team_phase_stats = deliveries_clean.groupby(
    ['match_id','season','batting_team','phase']
).agg(
    runs    = ('batsman_runs', 'sum'),
    wickets = ('is_wicket', 'sum'),
    balls   = ('ball', 'count'),
).reset_index()

team_phase_stats['run_rate'] = np.where(
    team_phase_stats['balls'] > 0,
    (team_phase_stats['runs'] / (team_phase_stats['balls']/6)).round(2), 0
)
print(f"team_phase_stats: {len(team_phase_stats)} rows")

# ═══════════════════════════════════════════════════════════════
# SECTION 3 — DIMENSION TABLES
# (These go on the OUTSIDE of the star — unique values only)
# RULE: Every dimension table must have NO duplicate values
#       in its key column. This guarantees one-to-many relationships.
# ═══════════════════════════════════════════════════════════════
print("\n── SECTION 3: BUILDING DIMENSION TABLES ──")

# ── DIM 1: SEASON ────────────────────────────────────────────
# One row per season — the slicer in Power BI will use this
dim_season = pd.DataFrame({
    'season'    : sorted(batter_stats['season'].dropna().unique()),
}).reset_index(drop=True)
dim_season['season_id'] = range(1, len(dim_season)+1)
# Confirm uniqueness
assert dim_season['season'].nunique() == len(dim_season), "season not unique!"
print(f"dim_season: {len(dim_season)} rows — {list(dim_season['season'])}")

# ── DIM 2: TEAM ──────────────────────────────────────────────
# One row per team — covers both batting and bowling teams
all_teams = pd.concat([
    batter_stats[['batting_team']].rename(columns={'batting_team':'team_name'}),
    bowler_stats[['bowling_team']].rename(columns={'bowling_team':'team_name'})
]).dropna().drop_duplicates().sort_values('team_name').reset_index(drop=True)
all_teams['team_id'] = range(1, len(all_teams)+1)
assert all_teams['team_name'].nunique() == len(all_teams), "team_name not unique!"
print(f"dim_team: {len(all_teams)} rows")

# ── DIM 3: VENUE ─────────────────────────────────────────────
# One row per venue
dim_venue = pd.DataFrame({
    'venue' : sorted(deliveries_clean['venue'].dropna().unique())
}).reset_index(drop=True)
dim_venue['venue_id'] = range(1, len(dim_venue)+1)

# Add useful venue info — avg first innings score
venue_avg = deliveries_clean[deliveries_clean['inning']==1].groupby('venue').agg(
    avg_first_innings_score = ('batsman_runs','sum'),
    total_matches           = ('match_id','nunique')
).reset_index()
venue_avg['avg_first_innings_score'] = (
    venue_avg['avg_first_innings_score'] / venue_avg['total_matches']
).round(1)
dim_venue = dim_venue.merge(venue_avg, on='venue', how='left')
assert dim_venue['venue'].nunique() == len(dim_venue), "venue not unique!"
print(f"dim_venue: {len(dim_venue)} rows")

# ── DIM 4: PLAYER ────────────────────────────────────────────
# One row per player — with career stats embedded
# Career batting stats
career_bat = batter_stats.groupby('batter').agg(
    career_runs      = ('runs','sum'),
    career_bat_matches = ('match_id','nunique'),
    career_avg       = ('runs','mean'),
    career_sr        = ('strike_rate','mean'),
    career_fifties   = ('is_fifty','sum'),
    career_hundreds  = ('is_hundred','sum'),
    career_bat_pts   = ('fantasy_pts','mean'),
).round(2).reset_index().rename(columns={'batter':'player_name'})

# Career bowling stats
career_bowl = bowler_stats.groupby('bowler').agg(
    career_wickets     = ('wickets','sum'),
    career_bowl_matches= ('match_id','nunique'),
    career_economy     = ('economy','mean'),
    career_bowl_pts    = ('fantasy_pts','mean'),
).round(2).reset_index().rename(columns={'bowler':'player_name'})

# Merge — full outer so all-rounders get both
dim_player = career_bat.merge(career_bowl, on='player_name', how='outer')
dim_player['player_id'] = range(1, len(dim_player)+1)

# Role — based on what data exists for them
dim_player['role'] = 'Unknown'
dim_player.loc[
    dim_player['career_runs'].notna() & dim_player['career_wickets'].isna(),
    'role'] = 'Batter'
dim_player.loc[
    dim_player['career_wickets'].notna() & dim_player['career_runs'].isna(),
    'role'] = 'Bowler'
dim_player.loc[
    dim_player['career_runs'].notna() & dim_player['career_wickets'].notna(),
    'role'] = 'All-rounder'
dim_player = dim_player.fillna(0)
assert dim_player['player_name'].nunique() == len(dim_player), "player not unique!"
print(f"dim_player: {len(dim_player)} rows")

# ── DIM 5: MATCH ─────────────────────────────────────────────
# One row per match — the bridge between all facts
dim_match = matches_clean[[
    'id','season','date','venue','team1','team2',
    'winner','toss_winner','toss_decision','toss_won_match'
]].copy().rename(columns={'id':'match_id'})
dim_match = dim_match.drop_duplicates(subset='match_id').reset_index(drop=True)
assert dim_match['match_id'].nunique() == len(dim_match), "match_id not unique!"
print(f"dim_match: {len(dim_match)} rows")

# ═══════════════════════════════════════════════════════════════
# SECTION 4 — SAVE ALL FILES
# ═══════════════════════════════════════════════════════════════
print("\n── SECTION 4: SAVING ALL FILES ──")
os.makedirs('../data/processed', exist_ok=True)

saves = {
    'fact_batter_stats'    : batter_stats,
    'fact_bowler_stats'    : bowler_stats,
    'fact_team_phase_stats': team_phase_stats,
    'dim_season'           : dim_season,
    'dim_team'             : all_teams,
    'dim_venue'            : dim_venue,
    'dim_player'           : dim_player,
    'dim_match'            : dim_match,
}

conn = sqlite3.connect('../data/processed/ipl.db')
for filename, df in saves.items():
    df.to_csv(f'../data/processed/{filename}.csv', index=False)
    df.to_sql(filename, conn, if_exists='replace', index=False)
    print(f"✓ {filename}.csv — {len(df)} rows")
conn.close()

# ═══════════════════════════════════════════════════════════════
# SECTION 5 — VALIDATE RELATIONSHIPS BEFORE POWER BI
# This checks that every join will be one-to-many — no errors
# ═══════════════════════════════════════════════════════════════
print("\n── SECTION 5: RELATIONSHIP VALIDATION ──")
checks = [
    ("dim_season[season]",   dim_season['season'],
     "fact_batter_stats[season]", batter_stats['season']),
    ("dim_season[season]",   dim_season['season'],
     "fact_bowler_stats[season]", bowler_stats['season']),
    ("dim_team[team_name]",  all_teams['team_name'],
     "fact_batter_stats[batting_team]", batter_stats['batting_team']),
    ("dim_team[team_name]",  all_teams['team_name'],
     "fact_bowler_stats[bowling_team]", bowler_stats['bowling_team']),
    ("dim_venue[venue]",     dim_venue['venue'],
     "fact_batter_stats[venue]", batter_stats['venue']),
    ("dim_match[match_id]",  dim_match['match_id'],
     "fact_batter_stats[match_id]", batter_stats['match_id']),
    ("dim_match[match_id]",  dim_match['match_id'],
     "fact_bowler_stats[match_id]", bowler_stats['match_id']),
]

all_good = True
for dim_label, dim_col, fact_label, fact_col in checks:
    dim_unique  = dim_col.nunique()
    fact_unique = fact_col.nunique()
    orphans = set(fact_col.unique()) - set(dim_col.unique())
    status = "✓ ONE-TO-MANY" if (dim_unique==len(dim_col) and len(orphans)==0) else "✗ ERROR"
    if status != "✓ ONE-TO-MANY":
        all_good = False
    print(f"{status} | {dim_label} ({dim_unique} unique) → {fact_label} ({len(fact_col)} rows) | orphans: {len(orphans)}")

print("\n" + ("✅ ALL RELATIONSHIPS VALID — SAFE TO BUILD IN POWER BI" if all_good else "❌ FIX ERRORS ABOVE BEFORE POWER BI"))

print("\n" + "="*60)
print("POWER BI LOAD ORDER & RELATIONSHIPS")
print("="*60)
print("""
STEP 1 — Load files in Power BI (Get Data → Text/CSV):
  dim_season.csv
  dim_team.csv
  dim_venue.csv
  dim_player.csv
  dim_match.csv
  fact_batter_stats.csv
  fact_bowler_stats.csv
  fact_team_phase_stats.csv

STEP 2 — In Power Query, for EVERY table:
  Select 'season' column → Data Type → Text
  Select 'date' column   → Data Type → Date
  Close & Apply

STEP 3 — Model View, create ONLY these 7 relationships:
  dim_season[season]    → fact_batter_stats[season]       (1:*)
  dim_season[season]    → fact_bowler_stats[season]       (1:*)
  dim_season[season]    → fact_team_phase_stats[season]   (1:*)
  dim_team[team_name]   → fact_batter_stats[batting_team] (1:*)
  dim_team[team_name]   → fact_bowler_stats[bowling_team] (1:*)
  dim_venue[venue]      → fact_batter_stats[venue]        (1:*)
  dim_match[match_id]   → fact_batter_stats[match_id]     (1:*)

STEP 4 — DO NOT connect:
  ✗ fact tables to each other
  ✗ dim_venue to dim_match (ambiguous path error)
  ✗ dim_match to fact_bowler_stats (not needed)
""")

IPL INTELLIGENCE PLATFORM — MASTER BUILD SCRIPT

── SECTION 1: LOADING RAW DATA ──
matches: 1095 rows | deliveries: 260920 rows
matches_clean: 1090 rows (removed 5 abandoned)
deliveries_clean: 260430 rows

── SECTION 2: BUILDING FACT TABLES ──
batter_stats: 16475 rows
bowler_stats: 12942 rows
team_phase_stats: 6389 rows

── SECTION 3: BUILDING DIMENSION TABLES ──
dim_season: 17 rows — ['2007-08', '2009', '2009-10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020-21', '2021', '2022', '2023', '2024']
dim_team: 14 rows
dim_venue: 58 rows
dim_player: 731 rows
dim_match: 1090 rows

── SECTION 4: SAVING ALL FILES ──
✓ fact_batter_stats.csv — 16475 rows
✓ fact_bowler_stats.csv — 12942 rows
✓ fact_team_phase_stats.csv — 6389 rows
✓ dim_season.csv — 17 rows
✓ dim_team.csv — 14 rows
✓ dim_venue.csv — 58 rows
✓ dim_player.csv — 731 rows
✓ dim_match.csv — 1090 rows

── SECTION 5: RELATIONSHIP VALIDATION ──
✓ ONE-TO-MANY | dim_season[season] (17 unique) → fact_batter_s

In [3]:
import pandas as pd
import os

# Current 10 IPL teams only (2024 season)
CURRENT_TEAMS = [
    'Mumbai Indians',
    'Chennai Super Kings', 
    'Royal Challengers Bangalore',
    'Kolkata Knight Riders',
    'Delhi Capitals',
    'Punjab Kings',
    'Rajasthan Royals',
    'Sunrisers Hyderabad',
    'Gujarat Titans',
    'Lucknow Super Giants'
]

base = '../data/processed/'

# Filter all fact tables to current teams only
batter = pd.read_csv(base + 'fact_batter_stats.csv')
bowler = pd.read_csv(base + 'fact_bowler_stats.csv')
phase  = pd.read_csv(base + 'fact_team_phase_stats.csv')
match  = pd.read_csv(base + 'dim_match.csv')

batter_filtered = batter[batter['batting_team'].isin(CURRENT_TEAMS)]
bowler_filtered = bowler[bowler['bowling_team'].isin(CURRENT_TEAMS)]
phase_filtered  = phase[phase['batting_team'].isin(CURRENT_TEAMS)]
match_filtered  = match[
    match['team1'].isin(CURRENT_TEAMS) & 
    match['team2'].isin(CURRENT_TEAMS)
]

print(f"Batter rows: {len(batter)} → {len(batter_filtered)}")
print(f"Bowler rows: {len(bowler)} → {len(bowler_filtered)}")
print(f"Phase rows:  {len(phase)} → {len(phase_filtered)}")
print(f"Match rows:  {len(match)} → {len(match_filtered)}")
print(f"\nSeasons available: {sorted(batter_filtered['season'].unique())}")
print(f"Teams: {sorted(batter_filtered['batting_team'].unique())}")

# Save filtered versions
batter_filtered.to_csv(base + 'fact_batter_stats.csv', index=False)
bowler_filtered.to_csv(base + 'fact_bowler_stats.csv', index=False)
phase_filtered.to_csv(base + 'fact_team_phase_stats.csv', index=False)
match_filtered.to_csv(base + 'dim_match.csv', index=False)

print("\n✓ All files updated with current teams only")

Batter rows: 16475 → 15433
Bowler rows: 12942 → 12115
Phase rows:  6389 → 5999
Match rows:  1090 → 961

Seasons available: ['2007-08', '2009', '2009-10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020-21', '2021', '2022', '2023', '2024']
Teams: ['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bangalore', 'Sunrisers Hyderabad']

✓ All files updated with current teams only
